In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset = pd.read_csv('./data/train_processed.csv')

X = dataset.drop('Attrition', axis=1)
y = dataset['Attrition']
X.head()

,Age,Gender,Years at Company,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,Distance from Home,Education Level,...,Job Role_Technology,Marital Status_Divorced,Marital Status_Married,Marital Status_Single,Overtime_No,Overtime_Yes,Leadership Opportunities_No,Leadership Opportunities_Yes,Innovation Opportunities_No,Innovation Opportunities_Yes
0,31,0,19,5390,3,1,2,2,22,1,...,False,False,True,False,True,False,True,False,True,False
1,59,1,4,5534,0,2,0,3,21,3,...,False,True,False,False,True,False,True,False,True,False
2,24,1,10,8159,2,2,0,0,11,2,...,False,False,True,False,True,False,True,False,True,False
3,36,1,7,3989,2,2,3,1,27,0,...,False,False,False,True,True,False,True,False,True,False
4,56,0,41,4821,1,3,2,0,71,0,...,False,True,False,False,False,True,True,False,True,False


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# model = []
folds = [4, 5, 6]

# def calc_metrics(X_train, X_valid, y_train, y_valid, model):
#   train_pred = model.predict_proba(X_train)[:,1]
#   valid_pred = model.predict_proba(X_valid)[:,1]

#   mse_train = mean_squared_error(y_train, train_pred)
#   r2_train = r2_score(y_train, train_pred)
#   mse_valid = mean_squared_error(y_valid, valid_pred)
#   r2_valid = r2_score(y_valid, valid_pred)

#   arr = np.array([[mse_train, r2_train], [mse_valid, r2_valid]])
#   return arr

# def kfold(X, Y, model, splits, dim_reduction=None):
#   kf = KFold(splits, shuffle=True, random_state=42)
#   metrics = np.zeros((2, 2))
#   for train_indices, valid_indices in kf.split(X, Y):

#     X_train, X_valid, Y_train, Y_valid = [X.iloc[train_indices], X.iloc[valid_indices], Y[train_indices], Y[valid_indices]]
#     if dim_reduction != None:
#       sc = StandardScaler()
#       X_train = dim_reduction.fit_transform(sc.fit_transform(X_train))
#       X_valid = dim_reduction.transform(sc.transform(X_valid))
#     model.fit(X_train, Y_train)
#     metrics += calc_metrics(X_train, X_valid, Y_train, Y_valid, model)

#   metrics /= splits   
#   print(model) 
#   display(pd.DataFrame([['train', metrics[0,0], metrics[0,1]], ['validation', metrics[1,0], metrics[1,1]]], columns=['set', 'MSE', 'R2']))
#   return model
  
# hidden_layer = (32,)

In [12]:
def get_model():
    model = Sequential()
    model.add(Dense(units=256, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=16, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=16, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=16, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae', 'r2_score'])
    return model

def get_model_t():
    model = Sequential()
    model.add(Dense(units=32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=256, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(optimizer='adam', 
                    loss='mse', 
                    metrics=['mae', 'r2_score'])
    return model

In [13]:
for f in folds:
  print(f'------------------------------------{f} folds------------------------------------')
  # model.append(kfold(X, y, LogisticRegression(max_iter=1000), f))
  # kfold with keras model

  kf = KFold(n_splits=f, shuffle=True, random_state=42)
  metrics = np.zeros((2, 2))
  for train_indices, val_indices in kf.split(X):
    # Tách dữ liệu và chuyển đổi thành kiểu số thực (float)
    X_train, X_val = X.iloc[train_indices].values.astype(int), X.iloc[val_indices].values.astype(int)
    y_train, y_val = y.iloc[train_indices].values.astype(int), y.iloc[val_indices].values.astype(int)

    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_val = sc.transform(X_val)

    # Create and train the model
    model = get_model()
    history = model.fit(X_train, y_train, 
                        epochs=20, 
                        batch_size=32, 
                        validation_data=(X_val, y_val), 
                        verbose=0)

    # Training and validation metrics
    train_loss, train_mae, train_r2= model.evaluate(X_train, y_train, verbose=0)
    val_loss, val_mae, val_r2 = model.evaluate(X_val, y_val, verbose=0)
    metrics += np.array([[train_mae, train_r2], [val_mae, val_r2]])
  # print average metrics
  metrics /= f
  display(pd.DataFrame([['train', metrics[0,0], metrics[0,1]], ['validation', metrics[1,0], metrics[1,1]]], columns=['set', 'mae', 'r2']))

------------------------------------4 folds------------------------------------


,set,mae,r2
0,train,0.312113,0.349707
1,validation,0.317421,0.329116


------------------------------------5 folds------------------------------------


,set,mae,r2
0,train,0.313763,0.347415
1,validation,0.318295,0.328672


------------------------------------6 folds------------------------------------


,set,mae,r2
0,train,0.325817,0.337952
1,validation,0.329868,0.323061


In [14]:
from sklearn.decomposition import PCA
for f in folds:
  print(f'------------------------------------{f} folds------------------------------------')
  # model.append(kfold(pd.DataFrame(X), y, LogisticRegression(max_iter=1000), f, PCA(n_components=X.shape[1] // 3)))
  kf = KFold(n_splits=f, shuffle=True, random_state=42)
  metrics = np.zeros((2, 2))
  for train_indices, val_indices in kf.split(X):
    X_train, X_val = X.iloc[train_indices].values.astype(int), X.iloc[val_indices].values.astype(int)
    y_train, y_val = y.iloc[train_indices].values.astype(int), y.iloc[val_indices].values.astype(int)

    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_val = sc.transform(X_val)
    pca = PCA()
    X_train = pca.fit_transform(X_train)
    X_val = pca.transform(X_val)

    # Create and train the model
    model = get_model_t()
    history = model.fit(X_train, y_train, 
                        epochs=20, 
                        batch_size=32, 
                        validation_data=(X_val, y_val), 
                        verbose=0)

    # Training and validation metrics
    train_loss, train_mae, train_r2= model.evaluate(X_train, y_train, verbose=0)
    val_loss, val_mae, val_r2 = model.evaluate(X_val, y_val, verbose=0)
    metrics += np.array([[train_mae, train_r2], [val_mae, val_r2]])
  # print average metrics
  metrics /= f
  display(pd.DataFrame([['train', metrics[0,0], metrics[0,1]], ['validation', metrics[1,0], metrics[1,1]]], columns=['set', 'mae', 'r2']))
  # worse perfomance seems to do with few features

------------------------------------4 folds------------------------------------


,set,mae,r2
0,train,0.404319,0.257078
1,validation,0.405568,0.251905


------------------------------------5 folds------------------------------------


,set,mae,r2
0,train,0.411905,0.252071
1,validation,0.413180,0.247068


------------------------------------6 folds------------------------------------


,set,mae,r2
0,train,0.415301,0.237710
1,validation,0.416283,0.233817
